In [ ]:
from pathlib import Path
import re
import random
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

In [ ]:
class TextCNN(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, num_filters=128, kernel_sizes=(3, 4, 5), dropout=0.5):
        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embed_dim,
            padding_idx=0
        )

        self.convs = nn.ModuleList([
            nn.Conv1d(
                in_channels=embed_dim,
                out_channels=num_filters,
                kernel_size=k
            )
            for k in kernel_sizes
        ])

        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(num_filters * len(kernel_sizes), 1)

    def forward(self, input_ids):
        embedded = self.embedding(input_ids)
        embedded = embedded.transpose(1, 2)

        conv_outputs = []
        for conv in self.convs:
            x = F.relu(conv(embedded))
            x = F.max_pool1d(x, kernel_size=x.shape[2]).squeeze(2)
            conv_outputs.append(x)

        x = torch.cat(conv_outputs, dim=1)
        x = self.dropout(x)
        logits = self.fc(x).squeeze(1)

        return logits

In [ ]:
p = Path("aclImdb")
data_dir = p

def load_imdb_folder(split):
    rows = []

    for label_name in ["pos", "neg"]:
        folder = data_dir / split / label_name

        if not folder.exists():
            raise FileNotFoundError(f"Could not find folder: {folder}")

        for file_path in folder.glob("*.txt"):
            text = file_path.read_text(encoding="utf-8", errors="ignore")
            label = "Positive" if label_name == "pos" else "Negative"

            rows.append({
                "Id": file_path.stem,
                "Review": text,
                "Label": label
            })

    return pd.DataFrame(rows)


train_df = load_imdb_folder("train")
test_df = load_imdb_folder("test")

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

display(train_df.head())
display(test_df.head())

print("Train labels:")
print(train_df["Label"].value_counts())

print("Test labels:")
print(test_df["Label"].value_counts())

In [ ]:
def clean_text(text):
    text = str(text)
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.lower().strip()


def tokenize(text):
    text = clean_text(text)
    return re.findall(r"[a-zA-Z0-9']+", text)

def build_vocab(texts, max_vocab_size=50000, min_freq=2):
    counter = Counter()

    for text in texts:
        counter.update(tokenize(text))

    vocab = {
        "<PAD>": 0,
        "<UNK>": 1
    }

    for word, freq in counter.most_common(max_vocab_size - 2):
        if freq >= min_freq:
            vocab[word] = len(vocab)

    return vocab

MAX_VOCAB_SIZE = 70000
MAX_LEN = 1024

vocab = build_vocab(
    train_df["Review"],
    max_vocab_size=MAX_VOCAB_SIZE,
    min_freq=2
)

print("Vocabulary size:", len(vocab))

In [ ]:
label_to_num = {
    "Negative": 0,
    "Positive": 1
}

num_to_label = {
    0: "Negative",
    1: "Positive"
}

train_data, val_data = train_test_split(
    train_df,
    test_size=0.2,
    random_state=SEED,
    stratify=train_df["Label"]
)

print("Train split:", train_data.shape)
print("Validation split:", val_data.shape)

print(train_data["Label"].value_counts())
print(val_data["Label"].value_counts())

In [ ]:
def encode_review(text, vocab, max_len=MAX_LEN):
    tokens = tokenize(text)
    ids = [vocab.get(token, vocab["<UNK>"]) for token in tokens]

    ids = ids[:max_len]

    if len(ids) < max_len:
        ids += [vocab["<PAD>"]] * (max_len - len(ids))

    return ids

class IMDBDataset(Dataset):
    def __init__(self, df, vocab, has_labels=True, max_len=MAX_LEN):
        self.df = df.reset_index(drop=True)
        self.vocab = vocab
        self.has_labels = has_labels
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        review = self.df.loc[idx, "Review"]
        input_ids = encode_review(review, self.vocab, self.max_len)
        input_ids = torch.tensor(input_ids, dtype=torch.long)

        if self.has_labels:
            label_text = self.df.loc[idx, "Label"]
            label = label_to_num[label_text]
            label = torch.tensor(label, dtype=torch.float)
            return input_ids, label

        return input_ids

In [ ]:
class TextCNN(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, num_filters=192, kernel_sizes=(2, 3, 4, 5), dropout=0.45):
        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embed_dim,
            padding_idx=0
        )

        self.convs = nn.ModuleList([
            nn.Conv1d(
                in_channels=embed_dim,
                out_channels=num_filters,
                kernel_size=k
            )
            for k in kernel_sizes
        ])

        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(num_filters * len(kernel_sizes), 1)

    def forward(self, input_ids):
        embedded = self.embedding(input_ids)
        embedded = embedded.transpose(1, 2)

        conv_outputs = []
        for conv in self.convs:
            x = F.relu(conv(embedded))
            x = F.max_pool1d(x, kernel_size=x.shape[2]).squeeze(2)
            conv_outputs.append(x)

        x = torch.cat(conv_outputs, dim=1)
        x = self.dropout(x)
        logits = self.fc(x).squeeze(1)
        return logits

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()

    total_loss = 0.0
    correct = 0
    total = 0

    for input_ids, labels in loader:
        input_ids = input_ids.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        logits = model(input_ids)
        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * labels.size(0)

        probs = torch.sigmoid(logits)
        preds = (probs >= 0.5).float()

        correct += (preds == labels).sum().item()
        total += labels.size(0)

    avg_loss = total_loss / total
    accuracy = correct / total

    return avg_loss, accuracy

def evaluate(model, loader, criterion, device):
    model.eval()

    total_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for input_ids, labels in loader:
            input_ids = input_ids.to(device)
            labels = labels.to(device)

            logits = model(input_ids)
            loss = criterion(logits, labels)

            total_loss += loss.item() * labels.size(0)

            probs = torch.sigmoid(logits)
            preds = (probs >= 0.5).float()

            correct += (preds == labels).sum().item()
            total += labels.size(0)

    avg_loss = total_loss / total
    accuracy = correct / total

    return avg_loss, accuracy

In [ ]:
BATCH_SIZE = 64
EPOCHS = 20
PATIENCE = 3

train_dataset = IMDBDataset(train_data, vocab, has_labels=True)
val_dataset = IMDBDataset(val_data, vocab, has_labels=True)
test_dataset = IMDBDataset(test_df, vocab, has_labels=True)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

model = TextCNN(
    vocab_size=len(vocab),
    embed_dim=128,
    num_filters=192,
    kernel_sizes=(2, 3, 4, 5, 7),
    dropout=0.45
).to(device)

num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("Trainable parameters:", num_params)

criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=5e-4,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=1
)

train_loss_values = []
val_loss_values = []
train_acc_values = []
val_acc_values = []

best_val_acc = 0.0
best_epoch = 0
epochs_without_improvement = 0

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = train_one_epoch(
        model,
        train_loader,
        optimizer,
        criterion,
        device
    )

    val_loss, val_acc = evaluate(
        model,
        val_loader,
        criterion,
        device
    )

    scheduler.step(val_acc)

    train_loss_values.append(train_loss)
    val_loss_values.append(val_loss)
    train_acc_values.append(train_acc)
    val_acc_values.append(val_acc)

    print(
        f"Epoch {epoch}/{EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.4f}"
    )

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_epoch = epoch
        epochs_without_improvement = 0
        torch.save(model.state_dict(), "best_textcnn_model.pt")
    else:
        epochs_without_improvement += 1

    if epochs_without_improvement >= PATIENCE:
        print(f"Early stopping at epoch {epoch}")
        break

model.load_state_dict(torch.load("best_textcnn_model.pt", map_location=device))
model.eval()

print("Best validation accuracy:", best_val_acc)
print("Best epoch:", best_epoch)

In [ ]:
epochs_ran = range(1, len(train_loss_values) + 1)

plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.semilogy(epochs_ran, train_loss_values, label="Train Loss")
plt.semilogy(epochs_ran, val_loss_values, label="Validation Loss")
plt.grid(True)
plt.title("Loss Values")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

plt.subplot(1, 3, 2)
plt.plot(epochs_ran, train_acc_values, label="Train Accuracy")
plt.hlines([0.8], 1, len(train_acc_values), colors="red", linestyles="dashed")
plt.ylim([0, 1])
plt.grid(True)
plt.title("Training Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()

plt.subplot(1, 3, 3)
plt.plot(epochs_ran, val_acc_values, label="Validation Accuracy")
plt.hlines([0.8], 1, len(val_acc_values), colors="red", linestyles="dashed")
plt.ylim([0, 1])
plt.grid(True)
plt.title("Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
def get_probs_and_labels(model, loader, device):
    model.eval()

    all_probs = []
    all_labels = []

    with torch.no_grad():
        for input_ids, labels in loader:
            input_ids = input_ids.to(device)

            logits = model(input_ids)
            probs = torch.sigmoid(logits)

            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(labels.numpy())

    return np.array(all_probs), np.array(all_labels)

val_probs, val_true = get_probs_and_labels(model, val_loader, device)

best_threshold = 0.5
best_val_acc = 0

for threshold in np.arange(0.30, 0.71, 0.01):
    val_preds = (val_probs >= threshold).astype(int)
    acc = (val_preds == val_true).mean()

    if acc > best_val_acc:
        best_val_acc = acc
        best_threshold = threshold

print("Best threshold:", best_threshold)
print("Best validation accuracy:", best_val_acc)

test_probs, test_true = get_probs_and_labels(model, test_loader, device)

test_preds = (test_probs >= best_threshold).astype(int)
test_acc = (test_preds == test_true).mean()

print("Test accuracy with tuned threshold:", test_acc)

In [ ]:
def predict_review_sentiment(review_text, threshold=best_threshold):
    model.eval()

    input_ids = encode_review(review_text, vocab)
    input_ids = torch.tensor(input_ids, dtype=torch.long).unsqueeze(0).to(device)

    with torch.no_grad():
        logits = model(input_ids)
        prob_positive = torch.sigmoid(logits).item()

    label = "Positive" if prob_positive >= threshold else "Negative"

    return label, prob_positive

my_review = "This movie has no right being this weird and ridiculous, but somehow I enjoyed every minute of it."

label, prob_positive = predict_review_sentiment(my_review, threshold=best_threshold)

print("Review:")
print(my_review)
print()
print("Threshold used:", best_threshold)
print("Prediction:", label)
print(f"Probability Positive: {prob_positive:.4f}")
print(f"Probability Negative: {1 - prob_positive:.4f}")

In [ ]:
def predict_labels(model, df, vocab, threshold=best_threshold):
    model.eval()

    predictions = []

    with torch.no_grad():
        for review in df["Review"]:
            input_ids = encode_review(review, vocab)
            input_ids = torch.tensor(input_ids, dtype=torch.long).unsqueeze(0).to(device)

            logits = model(input_ids)
            prob_positive = torch.sigmoid(logits).item()

            label = "Positive" if prob_positive >= threshold else "Negative"
            predictions.append(label)

    return predictions

test_predictions = predict_labels(
    model,
    test_df,
    vocab,
    threshold=best_threshold
)

submission = pd.DataFrame({
    "Id": test_df["Id"],
    "Label": test_predictions
})

submission.to_csv("prediction.csv", index=False)

display(submission.head())
print("Saved prediction.csv")
print("Threshold used:", best_threshold)